# ReFlex: live gRPC interaction extraction

This notebook observes the **existing Python backend**: incoming depth samples, the actual Butterworth filter output, and returned extrema. These are processor inputs and candidates, not unfiltered camera frames or the final calibrated ReFlex touch stream.

Run the cells in order. The default `observe` mode waits for the ReFlex tracking server. For a sensor-free demonstration, choose `demo` below **only while ReFlex processing is stopped**. Both producers use the same gRPC service and configuration; do not run them simultaneously.


In [ ]:
import asyncio
import json
import os
from pathlib import Path
import time

import grpc
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
from generated_services import interactionService_pb2_grpc as rpc
from generated_services import interactionTypes_pb2 as messages

MODE = "observe"  # "demo" sends synthetic samples; "observe" only reads snapshots.
DURATION_SECONDS = 60
REFRESH_SECONDS = 0.5
TARGET = os.getenv("GRPC_TARGET", "backend:50051")
SNAPSHOT = Path(os.getenv("SNAPSHOT_PATH", "/data/latest.json"))
assert MODE in {"observe", "demo"}


## Connection and optional synthetic input

`GetState` verifies the same endpoint used by `RemoteInteractionProcessingService`. In demo mode, `Configure` sets a cutoff of 2 and a sampling factor of 100, and `ComputeInteractions` sends a moving profile. Observe mode does not change the backend configuration. The channel is closed after each RPC, so rerunning cells does not leak connections.


In [ ]:
OPTIONS = [("grpc.max_receive_message_length", 8 * 1024 * 1024),
           ("grpc.max_send_message_length", 8 * 1024 * 1024)]

def call_rpc(method, request):
    with grpc.insecure_channel(TARGET, options=OPTIONS) as channel:
        return getattr(rpc.InteractionProcessingStub(channel), method)(request, timeout=3)

state = call_rpc("GetState", messages.StateRequest())
print(f"Ready: {state.isReady}; last frame: {state.frame_id}; mode: {MODE}")
if MODE == "demo":
    result = call_rpc("Configure", messages.ConfigRequest(cutoff=2.0, factor=100.0))
    assert result.success

def send_demo_frame(elapsed):
    x = np.linspace(0, 8 * np.pi, 1024)
    z = 1.0 + 0.03 * np.sin(x + elapsed) + 0.002 * np.sin(37 * x)
    return call_rpc("ComputeInteractions", messages.DepthValues1d(z=z.tolist()))


## Live view

The loop runs for one minute; rerun this cell for another interval, or stop immediately with **Kernel → Interrupt Kernel**. Only the latest completed frame is read; slower rendering skips frames. The frame age remains visible when the producer stops. Backend restarts are retried, and invalid demo requests are reported.

The wire format has no grid dimensions, X/Y coordinates or validity mask. All plots therefore use the **transmitted array index**. With `CompleteDataSet=false`, ReFlex maps these compact indices back to the point cloud. Do not reshape this array into a camera image without supplying the dimensions and matching mapping separately.


In [ ]:
started = time.monotonic()
try:
    while time.monotonic() - started < DURATION_SECONDS:
        try:
            if MODE == "demo":
                send_demo_frame(time.monotonic() - started)
            snapshot = json.loads(SNAPSHOT.read_text(encoding="utf-8"))
            age = max(0.0, time.time() - snapshot["received_at"])
            raw = np.asarray(snapshot["raw"])
            filtered = np.asarray(snapshot["filtered"])
            fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
            axes[0].plot(raw, linewidth=0.8)
            axes[0].set_title("Input: transmitted depth values")
            axes[1].plot(filtered, color="darkorange")
            axes[1].set_title(f"Butterworth low-pass: {snapshot['config']}")
            axes[2].plot(raw, color="gray", alpha=0.5, linewidth=0.8)
            for kind, color in [("PUSH", "crimson"), ("PULL", "royalblue")]:
                hits = [i for i in snapshot["interactions"] if i["type"] == kind]
                axes[2].scatter([i["point_idx"] for i in hits], [i["z"] for i in hits],
                                color=color, label=f"{kind}: {len(hits)}", zorder=3)
            axes[2].legend()
            axes[2].set_title("Returned candidates (original depth at each detected index)")
            axes[2].set_xlabel("Transmitted array index")
            for ax in axes:
                ax.set_ylabel("Sensor depth")
                ax.grid(alpha=0.2)
            status = "STALE / producer stopped?" if age > 3 else "receiving"
            fig.suptitle(f"Frame {snapshot['frame_id']} | age {age:.1f}s | {status}")
            fig.tight_layout()
            clear_output(wait=True)
            display(fig)
            plt.close(fig)
            print(json.dumps(snapshot["interactions"][:10], indent=2))
        except FileNotFoundError:
            clear_output(wait=True)
            print("Waiting for a frame. Start ReFlex remote processing, or select demo mode.")
        except grpc.RpcError as error:
            clear_output(wait=True)
            print(f"gRPC {error.code().name}: {error.details()}; retrying…")
        await asyncio.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("Live view stopped.")
finally:
    plt.close("all")


## Interpretation and limitations

The backend applies an order-5 Butterworth filter and finds minima/maxima with prominence `0.0005`. Detection uses the filtered signal, while each returned `z` is taken from the original input. These are one-dimensional extrema; row boundaries and compacted invalid points can affect detection. The backend assigns touch IDs per frame and currently emits Unix milliseconds. ReFlex performs additional classification, depth conversion and tracking afterwards; the notebook does not show those downstream steps.

Snapshots contain only one frame and are overwritten atomically. They cost serialization and disk I/O per frame, so this adapter is intended for learning and debugging. The original gRPC protocol is unchanged. To persist your notebook edits, use Save; Compose keeps the notebook volume until explicitly removed.
